In [1]:
!pip install -U transformers gradio  pandas numpy

In [2]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import gradio as gr

In [4]:
MODEL_NAME = "nlptown/bert-base-multilingual-uncased-sentiment"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

model.safetensors: reconstructing file:   0%|          |  0.00B /  669MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [5]:
text = "It was good but couldve been better. Great"

tokens = tokenizer.encode(
    text,
    return_tensors="pt"
)

tokens

tensor([[  101, 10197, 10140, 12050, 10502, 12296, 10598, 10662, 16197,   119,
         11838,   102]])

In [6]:
result = model(tokens)

result.logits

tensor([[-2.7768, -1.2353,  1.4419,  1.9804,  0.4584]],
       grad_fn=<AddmmBackward0>)

In [7]:
score = int(torch.argmax(result.logits)) + 1

print("Sentiment:", score)

Sentiment: 4


In [8]:
def predict_sentiment(review):

    tokens = tokenizer.encode(
        review,
        return_tensors="pt"
    )

    with torch.no_grad():
        result = model(tokens)

    score = int(torch.argmax(result.logits)) + 1

    labels = {
        1: "Very Negative 😡",
        2: "Negative 🙁",
        3: "Neutral 😐",
        4: "Positive 🙂",
        5: "Very Positive 😍"
    }

    return f"{labels[score]} — Score: {score}/5"

In [9]:
predict_sentiment("I absolutely loved this movie!")

'Very Positive 😍 — Score: 5/5'

In [10]:
predict_sentiment("The food was terrible.")

'Very Negative 😡 — Score: 1/5'

In [11]:
interface = gr.Interface(
    fn=predict_sentiment,
    inputs=gr.Textbox(
        lines=5,
        placeholder="Write a review here...",
        label="Your Review"
    ),
    outputs=gr.Textbox(
        label="Sentiment"
    ),
    title="🤖 BERT Sentiment Analyzer",
    description="Enter a review and BERT will predict its sentiment from 1 to 5."
)

interface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6a7d8fae2a65b32870.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
